<a href="https://colab.research.google.com/github/carollinemelloo/AnaliseEProjeto/blob/main/Compiladores2bim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports e Analisador Léxico

In [ ]:
import re

In [ ]:
class Token:
    def __init__(self, type, value):
        self.type = type
        self.value = value

    def __repr__(self):
        return f"Token({self.type}, {self.value})"

class Lexer:
    def __init__(self, source_code):
        self.source = source_code
        self.tokens = []


        self.regex_patterns = [
            ('MAIN',    r'main'),
            ('IF',      r'if'),
            ('ELSE',    r'else'),
            ('TIPO',    r'(int|float|double)'),
            ('NUM',     r'\d+(\.\d+)?'),
            ('COMP',    r'(==|<=|>=|<|>)'),
            ('ID',      r'[a-zA-Z_][a-zA-Z0-9_]*'),
            ('ASSIGN',  r'='),
            ('PLUS',    r'\+'),
            ('MINUS',   r'-'),
            ('MULT',    r'\*'),
            ('DIV',     r'/'),
            ('LPAREN',  r'\('),
            ('RPAREN',  r'\)'),
            ('LBRACE',  r'\{'),
            ('RBRACE',  r'\}'),
            ('SEMI',    r';'),
            ('SKIP',    r'[ \t\n]+'),
            ('MISMATCH',r'.'),
        ]

    def tokenize(self):
        pos = 0
        while pos < len(self.source):
            match = None
            for token_type, pattern in self.regex_patterns:
                regex = re.compile(pattern)
                match = regex.match(self.source, pos)
                if match:
                    text = match.group(0)
                    if token_type == 'MISMATCH':
                        raise Exception(f"Erro Léxico: Caractere inesperado '{text}'")
                    elif token_type != 'SKIP':
                        term_map = {
                            'MAIN': 'main', 'IF': 'if', 'ELSE': 'else',
                            'ASSIGN': '=', 'PLUS': '+', 'MINUS': '-',
                            'MULT': '*', 'DIV': '/', 'LPAREN': '(',
                            'RPAREN': ')', 'LBRACE': '{', 'RBRACE': '}',
                            'SEMI': ';', 'ID': 'id', 'NUM': 'num',
                            'TIPO': 'type', 'COMP': 'comp'
                        }
                        gram_token = term_map.get(token_type, text)
                        self.tokens.append(Token(gram_token, text))

                    pos = match.end()
                    break
            if not match:
                raise Exception("Erro fatal no Scanner")

        self.tokens.append(Token('$', '$'))
        return self.tokens

print("Célula 1 executada: Scanner pronto.")

Célula 1 executada: Scanner pronto.


Gramática e Gerador LL(1)

In [ ]:
grammar = {
    'Program':    [['type', 'main', '(', ')', '{', 'Block', '}']],
    'Block':      [['Statement', 'Block'], ['eps']],
    'Statement':  [['Decl'], ['IfStmt'], ['Assign']],
    'Decl':       [['type', 'id', ';']],
    'Assign':     [['id', '=', 'Expr', ';']],
    'IfStmt':     [['if', '(', 'Logic', ')', '{', 'Block', '}', 'ElsePart']],
    'ElsePart':   [['else', '{', 'Block', '}'], ['eps']],
    'Expr':       [['Term', 'ExprPrime']],
    'ExprPrime':  [['+', 'Term', 'ExprPrime'], ['-', 'Term', 'ExprPrime'], ['eps']],
    'Term':       [['Factor', 'TermPrime']],
    'TermPrime':  [['*', 'Factor', 'TermPrime'], ['/', 'Factor', 'TermPrime'], ['eps']],
    'Factor':     [['(', 'Expr', ')'], ['id'], ['num']],
    'Logic':      [['Expr', 'comp', 'Expr']]
}

start_symbol = 'Program'
terminals = {'type', 'main', '(', ')', '{', '}', 'id', ';', '=', 'if', 'else', '+', '-', '*', '/', 'num', 'comp', '$'}
non_terminals = set(grammar.keys())

class LL1Generator:
    def __init__(self, grammar):
        self.grammar = grammar
        self.first = {nt: set() for nt in non_terminals}
        self.follow = {nt: set() for nt in non_terminals}
        self.table = {}

    def compute_first(self):
        changed = True
        while changed:
            changed = False
            for head, productions in self.grammar.items():
                for prod in productions:
                    if prod[0] == 'eps':
                        if 'eps' not in self.first[head]:
                            self.first[head].add('eps')
                            changed = True
                        continue
                    for symbol in prod:
                        symbol_first = {symbol} if symbol in terminals else self.first[symbol]
                        to_add = symbol_first - {'eps'}
                        if not to_add.issubset(self.first[head]):
                            self.first[head].update(to_add)
                            changed = True
                        if 'eps' not in symbol_first: break
                    else:
                        if 'eps' not in self.first[head]:
                            self.first[head].add('eps')
                            changed = True

    def compute_follow(self):
        self.follow[start_symbol].add('$')
        changed = True
        while changed:
            changed = False
            for head, productions in self.grammar.items():
                for prod in productions:
                    if prod[0] == 'eps': continue
                    trailer = self.follow[head].copy()
                    for symbol in reversed(prod):
                        if symbol in non_terminals:
                            if not trailer.issubset(self.follow[symbol]):
                                self.follow[symbol].update(trailer)
                                changed = True
                            if 'eps' in self.first[symbol]:
                                trailer.update(self.first[symbol] - {'eps'})
                            else:
                                trailer = self.first[symbol].copy()
                        else:
                            trailer = {symbol}

    def build_table(self):
        for head, productions in self.grammar.items():
            for prod in productions:
                first_alpha = set()
                if prod[0] == 'eps': first_alpha = {'eps'}
                elif prod[0] in terminals: first_alpha = {prod[0]}
                else: first_alpha = self.first[prod[0]]

                for term in first_alpha:
                    if term != 'eps': self.table[(head, term)] = prod
                if 'eps' in first_alpha:
                    for b in self.follow[head]: self.table[(head, b)] = prod

print("Célula 2 executada: Motor LL(1) configurado.")

Célula 2 executada: Motor LL(1) configurado.


Função de Parsing (Driver)

In [ ]:
def parse(tokens, table):
    stack = ['$', start_symbol]
    cursor = 0

    print(f"\n{'PILHA':<50} | {'ENTRADA':<30} | {'AÇÃO'}")
    print("-" * 100)

    while len(stack) > 0:
        top = stack[-1]
        current_token = tokens[cursor]
        token_type = current_token.type

        stack_str = " ".join(stack)
        input_str = " ".join([t.value for t in tokens[cursor:]])

        if top == '$' and token_type == '$':
            print(f"{stack_str:<50} | {input_str:<30} | ACEITO")
            return True
        elif top == token_type:
            print(f"{stack_str:<50} | {input_str:<30} | Match: {token_type}")
            stack.pop()
            cursor += 1
        elif (top, token_type) in table:
            production = table[(top, token_type)]
            print(f"{stack_str:<50} | {input_str:<30} | Deriva: {top} -> {' '.join(production)}")
            stack.pop()
            if production != ['eps']:
                for symbol in reversed(production):
                    stack.append(symbol)
        else:
            print(f"{stack_str:<50} | {input_str:<30} | ERRO")
            raise Exception(f"Erro Sintático: Esperado '{top}', encontrado '{current_token.value}'")

print("Célula 3 executada: Driver de Parsing pronto.")

Célula 3 executada: Driver de Parsing pronto.


Execução do Teste LL(1)

In [ ]:
codigo_fonte = """
int main() {
    int x;
    x = 10;
    if (x > 5) {
        x = x + 1;
    } else {
        x = 0;
    }
}
"""

print("--- CÓDIGO FONTE ---")
print(codigo_fonte)

try:
    lexer = Lexer(codigo_fonte)
    tokens = lexer.tokenize()
    print("\n--- TOKENS ---")
    print(tokens)

    gen = LL1Generator(grammar)
    gen.compute_first()
    gen.compute_follow()
    gen.build_table()

    print("\n--- PARSING LL(1) ---")
    parse(tokens, gen.table)
    print("\nSUCESSO: O código é válido na gramática LL(1)!")

except Exception as e:
    print(f"\nERRO: {e}")

--- CÓDIGO FONTE ---

int main() {
    int x;
    x = 10;
    if (x > 5) {
        x = x + 1;
    } else {
        x = 0;
    }
}


--- TOKENS ---
[Token(type, int), Token(main, main), Token((, (), Token(), )), Token({, {), Token(type, int), Token(id, x), Token(;, ;), Token(id, x), Token(=, =), Token(num, 10), Token(;, ;), Token(if, if), Token((, (), Token(id, x), Token(comp, >), Token(num, 5), Token(), )), Token({, {), Token(id, x), Token(=, =), Token(id, x), Token(+, +), Token(num, 1), Token(;, ;), Token(}, }), Token(else, else), Token({, {), Token(id, x), Token(=, =), Token(num, 0), Token(;, ;), Token(}, }), Token(}, }), Token($, $)]

--- PARSING LL(1) ---

PILHA                                              | ENTRADA                        | AÇÃO
----------------------------------------------------------------------------------------------------
$ Program                                          | int main ( ) { int x ; x = 10 ; if ( x > 5 ) { x = x + 1 ; } else { x = 0 ; } } $ | De

Testes de Erro

In [ ]:
print(">>> INICIO DO TESTE 1: Validacao de Erro Lexico <<<\n")

codigo_lex_err = "int main() { int x @; }"
print(f"Entrada fornecida:\n{codigo_lex_err}")
print("-" * 40)

try:
    l = Lexer(codigo_lex_err)
    t = l.tokenize()
    print("[FALHA] O analisador incorretamente aceitou a entrada com caracteres invalidos.")
except Exception as e:
    print(f"[SUCESSO] O analisador lexico identificou a anomalia corretamente.")
    print(f"Detalhe da excecao capturada: {e}")

>>> INICIO DO TESTE 1: Validacao de Erro Lexico <<<

Entrada fornecida:
int main() { int x @; }
----------------------------------------
[SUCESSO] O analisador lexico identificou a anomalia corretamente.
Detalhe da excecao capturada: Erro Léxico: Caractere inesperado '@'


In [ ]:
print(">>> INICIO DO TESTE 2: Validacao de Erro Sintatico (Gramatica) <<<\n")


codigo_syn_err = """
int main() {
    int x;
    x = 10
}
"""
print(f"Entrada fornecida:\n{codigo_syn_err}")
print("-" * 40)

try:
    l = Lexer(codigo_syn_err)
    t = l.tokenize()

    g = LL1Generator(grammar)
    g.compute_first()
    g.compute_follow()
    g.build_table()

    parse(t, g.table)
    print("[FALHA] O parser completou a execucao, mas deveria ter falhado.")
except Exception as e:
    print(f"[SUCESSO] O analisador sintatico interrompeu a execucao conforme esperado.")
    print(f"Detalhe da excecao capturada: {e}")

>>> INICIO DO TESTE 2: Validacao de Erro Sintatico (Gramatica) <<<

Entrada fornecida:

int main() {
    int x;
    x = 10
}

----------------------------------------

PILHA                                              | ENTRADA                        | AÇÃO
----------------------------------------------------------------------------------------------------
$ Program                                          | int main ( ) { int x ; x = 10 } $ | Deriva: Program -> type main ( ) { Block }
$ } Block { ) ( main type                          | int main ( ) { int x ; x = 10 } $ | Match: type
$ } Block { ) ( main                               | main ( ) { int x ; x = 10 } $  | Match: main
$ } Block { ) (                                    | ( ) { int x ; x = 10 } $       | Match: (
$ } Block { )                                      | ) { int x ; x = 10 } $         | Match: )
$ } Block {                                        | { int x ; x = 10 } $           | Match: {
$ } Block               

In [ ]:
print(">>> INICIO DO TESTE 3: Validacao Semantica (Variavel Nao Declarada) <<<\n")

def analise_semantica_simples(tokens):
    """
    Verifica se todas as variáveis usadas foram declaradas.
    Regra: Se aparece 'id', ele deve ter sido declarado antes.
    """
    tabela_simbolos = set()

    for i in range(len(tokens)):
        token = tokens[i]

        if token.type == 'id':
            token_anterior = tokens[i-1] if i > 0 else None

            if token_anterior and token_anterior.type == 'type':
                tabela_simbolos.add(token.value)

            else:
                if token.value not in tabela_simbolos:
                    raise Exception(f"Erro Semantico: A variavel '{token.value}' foi utilizada sem declaracao previa.")

codigo_semantico = """
int main() {
    int x;
    x = 10;
    y = 20;
}
"""
print(f"Entrada fornecida:\n{codigo_semantico}")
print("-" * 40)

try:
    l = Lexer(codigo_semantico)
    tokens = l.tokenize()

    analise_semantica_simples(tokens)
    print("[FALHA] O verificador semantico nao detectou a inconsistencia.")

except Exception as e:
    print(f"[SUCESSO] Violacao semantica detectada pelo analisador.")
    print(f"Detalhe da excecao capturada: {e}")

>>> INICIO DO TESTE 3: Validacao Semantica (Variavel Nao Declarada) <<<

Entrada fornecida:

int main() {
    int x;
    x = 10;
    y = 20; 
}

----------------------------------------
[SUCESSO] Violacao semantica detectada pelo analisador.
Detalhe da excecao capturada: Erro Semantico: A variavel 'y' foi utilizada sem declaracao previa.


Analisador SLR

Scanner e Estrutura SLR

In [ ]:
slr_grammar = [
    (0, 'S\'', ['S']), (1, 'S', ['id', '=', 'E']), (2, 'E', ['E', '+', 'T']),
    (3, 'E', ['T']), (4, 'T', ['T', '*', 'F']), (5, 'T', ['F']),
    (6, 'F', ['(', 'E', ')']), (7, 'F', ['id']), (8, 'F', ['num']),
]
slr_follow = {'S\'': {'$'}, 'S': {'$'}, 'E': {'+', ')', '$', ';'},
              'T': {'*', '+', ')', '$', ';'}, 'F': {'*', '+', ')', '$', ';'}}

class SLRGenerator:
    def __init__(self, rules):
        self.rules = rules
        self.terminals = {'id', 'num', '+', '*', '(', ')', '=', ';', '$'}
        self.non_terminals = {'S', 'E', 'T', 'F'}
        self.canonical_collection = []
        self.action = {}
        self.goto = {}

    def closure(self, items):
        closure_set = set(items)
        changed = True
        while changed:
            changed = False
            temp_add = set()
            for (rule_idx, dot_pos) in closure_set:
                rhs = self.rules[rule_idx][2]
                if dot_pos < len(rhs) and rhs[dot_pos] in self.non_terminals:
                    symbol = rhs[dot_pos]
                    for i, rule in enumerate(self.rules):
                        if rule[1] == symbol:
                            new_item = (i, 0)
                            if new_item not in closure_set and new_item not in temp_add:
                                temp_add.add(new_item)
                                changed = True
            closure_set.update(temp_add)
        return sorted(list(closure_set))

    def goto_op(self, items, symbol):
        goto_set = set()
        for (rule_idx, dot_pos) in items:
            rhs = self.rules[rule_idx][2]
            if dot_pos < len(rhs) and rhs[dot_pos] == symbol:
                goto_set.add((rule_idx, dot_pos + 1))
        return self.closure(list(goto_set))

    def build_table(self):
        start_item = self.closure([(0, 0)])
        self.canonical_collection = [start_item]
        changed = True
        while changed:
            changed = False
            for i, state_items in enumerate(self.canonical_collection):
                all_symbols = self.terminals.union(self.non_terminals)
                for symbol in all_symbols:
                    next_state = self.goto_op(state_items, symbol)
                    if not next_state: continue
                    if next_state not in self.canonical_collection:
                        self.canonical_collection.append(next_state)
                        changed = True

        for state_idx, items in enumerate(self.canonical_collection):
            for (rule_idx, dot_pos) in items:
                lhs, rhs = self.rules[rule_idx][1], self.rules[rule_idx][2]
                if dot_pos < len(rhs):
                    symbol = rhs[dot_pos]
                    if symbol in self.terminals:
                        next_state = self.goto_op(items, symbol)
                        if next_state in self.canonical_collection:
                            self.action[(state_idx, symbol)] = ('s', self.canonical_collection.index(next_state))
                else:
                    if lhs == 'S\'': self.action[(state_idx, '$')] = ('acc', 0)
                    else:
                        for f in slr_follow[lhs]: self.action[(state_idx, f)] = ('r', rule_idx)
            for nt in self.non_terminals:
                nxt = self.goto_op(items, nt)
                if nxt and nxt in self.canonical_collection:
                    self.goto[(state_idx, nt)] = self.canonical_collection.index(nxt)

print("Célula 5 executada: Motor SLR configurado.")

Célula 5 executada: Motor SLR configurado.


Execução do Teste SLR

In [ ]:
class SLRLexer:
    def __init__(self, source):
        self.source = source
        self.tokens = []
    def tokenize(self):
        patterns = [('id', r'[a-zA-Z_]\w*'), ('num', r'\d+'), ('+', r'\+'),
                    ('*', r'\*'), ('(', r'\('), (')', r'\)'), ('=', r'='), (';', r';')]
        pos = 0
        while pos < len(self.source):
            if self.source[pos].isspace():
                pos += 1; continue
            match = None
            for type, pat in patterns:
                m = re.match(pat, self.source[pos:])
                if m:
                    self.tokens.append(Token(type, m.group(0)))
                    pos += m.end()
                    match = True; break
            if not match: raise Exception("Erro Lexico SLR")
        self.tokens.append(Token('$', '$'))
        return self.tokens

def slr_parse(tokens, slr_gen):
    stack = [0]
    cursor = 0
    print(f"\n{'PILHA':<30} | {'ENTRADA':<20} | {'AÇÃO'}")
    print("-" * 80)
    while True:
        state = stack[-1]
        token_type = tokens[cursor].type
        action = slr_gen.action.get((state, token_type))

        stack_str, input_str = str(stack), " ".join([t.value for t in tokens[cursor:]])

        if not action:
            print(f"{stack_str:<30} | {input_str:<20} | ERRO"); return False

        act, val = action
        if act == 's':
            print(f"{stack_str:<30} | {input_str:<20} | Shift {val}")
            stack.append(val); cursor += 1
        elif act == 'r':
            rule = slr_gen.rules[val]
            print(f"{stack_str:<30} | {input_str:<20} | Reduce {rule[1]}->{' '.join(rule[2])}")
            for _ in rule[2]: stack.pop()
            stack.append(slr_gen.goto[(stack[-1], rule[1])])
        elif act == 'acc':
            print(f"{stack_str:<30} | {input_str:<20} | ACEITO!"); return True

codigo_slr = "x = y * 10"
print(f"--- TESTE SLR: {codigo_slr} ---")
lexer = SLRLexer(codigo_slr)
tokens = lexer.tokenize()
slr = SLRGenerator(slr_grammar)
slr.build_table()
slr_parse(tokens, slr)

--- TESTE SLR: x = y * 10 ---

PILHA                          | ENTRADA              | AÇÃO
--------------------------------------------------------------------------------
[0]                            | x = y * 10 $         | Shift 2
[0, 2]                         | = y * 10 $           | Shift 3
[0, 2, 3]                      | y * 10 $             | Shift 9
[0, 2, 3, 9]                   | * 10 $               | Reduce F->id
[0, 2, 3, 4]                   | * 10 $               | Reduce T->F
[0, 2, 3, 6]                   | * 10 $               | Shift 11
[0, 2, 3, 6, 11]               | 10 $                 | Shift 8
[0, 2, 3, 6, 11, 8]            | $                    | Reduce F->num
[0, 2, 3, 6, 11, 14]           | $                    | Reduce T->T * F
[0, 2, 3, 6]                   | $                    | Reduce E->T
[0, 2, 3, 7]                   | $                    | Reduce S->id = E
[0, 1]                         | $                    | ACEITO!


True

Testes de Erro

In [ ]:
print(">>> INICIO DO TESTE 4: Validacao de Erros no Algoritmo SLR <<<\n")

print("--- Caso A: Teste de Lexema Invalido ---")
try:
    lexer_slr = SLRLexer("x = y ? 10")
    t = lexer_slr.tokenize()
    print("[FALHA] O Lexer SLR aceitou caracteres invalidos.")
except Exception as e:
     print(f"[SUCESSO] Erro lexico capturado corretamente: {e}")

print("\n" + "-" * 40 + "\n")

print("--- Caso B: Teste de Ordem Sintatica Invalida ---")
try:
    lexer_slr = SLRLexer("x = + 10")
    t = lexer_slr.tokenize()

    slr_gen = SLRGenerator(slr_grammar)
    slr_gen.build_table()

    print("Iniciando analise SLR...")
    resultado = slr_parse(t, slr_gen)

    if not resultado:
        print("\n[SUCESSO] O algoritmo SLR rejeitou a entrada e interrompeu o processamento.")
    else:
        print("[FALHA] O algoritmo aceitou uma entrada sintaticamente invalida.")
except Exception as e:
    print(f"Erro de execucao nao tratado: {e}")

>>> INICIO DO TESTE 4: Validacao de Erros no Algoritmo SLR <<<

--- Caso A: Teste de Lexema Invalido ---
[SUCESSO] Erro lexico capturado corretamente: Erro Lexico SLR

----------------------------------------

--- Caso B: Teste de Ordem Sintatica Invalida ---
Iniciando analise SLR...

PILHA                          | ENTRADA              | AÇÃO
--------------------------------------------------------------------------------
[0]                            | x = + 10 $           | Shift 2
[0, 2]                         | = + 10 $             | Shift 3
[0, 2, 3]                      | + 10 $               | ERRO

[SUCESSO] O algoritmo SLR rejeitou a entrada e interrompeu o processamento.
